In [76]:
import os
import numpy as np
import pandas as pd
data_filename = 'Ch3_NBA_2013_2014_Games_csv.format.txt'
dataset = pd.read_csv(data_filename)
#print(dataset)
#dataset.loc[:5]  # labels
dataset.iloc[:5]   # first 5 rows

,Date,Visitor,VisitorPts,Home,HomePts
0,10/30/2013,ORL,87,IND,97
1,10/30/2013,LAC,103,LAL,116
2,10/30/2013,CHI,95,MIA,107
3,10/30/2013,BRK,94,CLE,98
4,10/30/2013,ATL,109,DAL,118


In [68]:
dataset["HomeWin"] = dataset["VisitorPts"] < dataset["HomePts"]
y_true = dataset["HomeWin"].values

In [69]:
#dataset = pd.read_csv('Ch3_NBA_2013_2014_Games_csv.format.txt', parse_dates=["Date"])
#dataset.columns = ["Date", "Score Type", "Visitor Team",
#"VisitorPts", "Home Team", "HomePts", "OT?", "Notes"]

In [70]:
from collections import defaultdict
won_last = defaultdict(int)
dataset["HomeLastWin"] = 0
dataset["VisitorLastWin"] = 0
for index, row in dataset.iterrows():
    home_team = row["Home"]
    visitor_team = row["Visitor"]
    row["HomeLastWin"] = won_last[home_team]
    row["VisitorLastWin"] = won_last[visitor_team]
    dataset.loc[index] = row

    won_last[home_team] = row["HomeWin"]
    won_last[visitor_team] = not row["HomeWin"]

#dataset.iloc[20:25] 
dataset.head()

/var/folders/25/ykptbny53096srygl9_fszkm0000gn/T/ipykernel_5531/1034098750.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataset.loc[index] = row
/var/folders/25/ykptbny53096srygl9_fszkm0000gn/T/ipykernel_5531/1034098750.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataset.loc[index] = row


,Date,Visitor,VisitorPts,Home,HomePts,HomeWin,HomeLastWin,VisitorLastWin
0,10/30/2013,ORL,87,IND,97,True,0,0
1,10/30/2013,LAC,103,LAL,116,True,0,0
2,10/30/2013,CHI,95,MIA,107,True,0,0
3,10/30/2013,BRK,94,CLE,98,True,0,0
4,10/30/2013,ATL,109,DAL,118,True,0,0


In [71]:
#Using Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
clf = DecisionTreeClassifier(random_state=14)
X_previouswins = dataset[["HomeLastWin", "VisitorLastWin"]].values
scores = cross_val_score(clf, X_previouswins, y_true,
scoring='accuracy')
print("Accuracy: {0:.1f}%".format(np.mean(scores) * 100))

Accuracy: 59.2%


In [90]:
#expanded standings
standings_filename = os.path.join(
"Ch3_NBA_2013_Games_ExpandedStanding_csv.format.txt")
standings = pd.read_csv(standings_filename, skiprows=[0])

standings

,Rk,Team,Overall,Home,Road,E,W,A,C,SE,...,Post,≤3,≥10,Oct,Nov,Dec,Jan,Feb,Mar,Apr
0,1,Miami Heat,66-16,37-4,29-12,41-11,25-5,14-4,12-6,15-1,...,30-2,9-3,39-8,1-0,10-3,10-5,8-5,12-1,17-1,8-1
1,2,Oklahoma City Thunder,60-22,34-7,26-15,21-9,39-13,7-3,8-2,6-4,...,21-8,3-6,44-6,NaN,13-4,11-2,11-5,7-4,12-5,6-2
2,3,San Antonio Spurs,58-24,35-6,23-18,25-5,33-19,8-2,9-1,8-2,...,16-12,9-5,31-10,1-0,12-4,12-4,12-3,8-3,10-4,3-6
3,4,Denver Nuggets,57-25,38-3,19-22,19-11,38-14,5-5,10-0,4-6,...,24-4,11-7,28-8,0-1,8-8,9-6,12-3,8-4,13-2,7-1
4,5,Los Angeles Clippers,56-26,32-9,24-17,21-9,35-17,7-3,8-2,6-4,...,17-9,3-5,38-12,1-0,8-6,16-0,9-7,8-5,7-7,7-1
5,6,Memphis Grizzlies,56-26,32-9,24-17,22-8,34-18,8-2,8-2,6-4,...,23-8,6-4,28-9,0-1,12-1,7-7,10-7,9-2,11-6,7-2
6,7,New York Knicks,54-28,31-10,23-18,37-15,17-13,10-6,12-6,15-3,...,22-10,7-5,31-12,NaN,11-4,10-5,7-6,6-5,12-6,8-2
7,8,Brooklyn Nets,49-33,26-15,23-18,36-16,13-17,11-5,13-5,12-6,...,18-11,9-4,23-17,NaN,11-4,5-11,11-4,7-5,8-7,7-2
8,9,Indiana Pacers,49-32,30-11,19-21,31-20,18-12,6-11,13-3,12-6,...,17-11,4-9,27-14,1-0,7-8,10-5,9-6,9-3,11-5,2-5
9,10,Golden State Warriors,47-35,28-13,19-22,19-11,28-24,7-3,5-5,7-3,...,17-13,5-3,20-18,1-0,8-6,12-4,8-7,4-8,9-7,5-3


In [92]:
dataset["HomeTeamRanksHigher"] = 0
for index, row in dataset.iterrows():
    home_team = row["Home"].strip()
    visitor_team = row["Visitor"].strip()
    
    # Handle team name changes
    if home_team == "New Orleans Pelicans":
        home_team = "New Orleans Hornets"
    if visitor_team == "New Orleans Pelicans":
        visitor_team = "New Orleans Hornets"
    
    # Strip spaces from standings names too
    standings["Team"] = standings["Team"].str.strip()
    
    # Get home rank safely
    home_row = standings[standings["Team"] == home_team]
    visitor_row = standings[standings["Team"] == visitor_team]
    
    if len(home_row) == 0 or len(visitor_row) == 0:
        # If team not found, assign default rank or NaN
        home_rank = visitor_rank = np.nan
        print(f"Team not found: {home_team} or {visitor_team}")
    else:
        home_rank = home_row["Rk"].values[0]
        visitor_rank = visitor_row["Rk"].values[0]
    
    # Add new column safely
    dataset.at[index, "HomeTeamRanksHigher"] = int(home_rank > visitor_rank) if not np.isnan(home_rank) else np.nan

Team not found: IND or ORL
Team not found: LAL or LAC
Team not found: MIA or CHI
Team not found: CLE or BRK
Team not found: DAL or ATL
Team not found: DET or WAS
Team not found: GSW or LAL
Team not found: HOU or CHA
Team not found: MIN or ORL
Team not found: NOP or IND
Team not found: NYK or MIL
Team not found: PHI or MIA
Team not found: PHO or POR
Team not found: SAC or DEN
Team not found: SAS or MEM
Team not found: TOR or BOS
Team not found: UTA or OKC
Team not found: CHI or NYK
Team not found: LAC or GSW
Team not found: ATL or TOR
Team not found: BOS or MIL
Team not found: BRK or MIA
Team not found: CHA or CLE
Team not found: DEN or POR
Team not found: HOU or DAL
Team not found: LAL or SAS
Team not found: MEM or DET
Team not found: MIN or OKC
Team not found: ORL or NOP
Team not found: PHO or UTA
Team not found: SAC or LAC
Team not found: WAS or PHI
Team not found: DAL or MEM
Team not found: GSW or SAC
Team not found: IND or CLE
Team not found: MIL or TOR
Team not found: NOP or CHA
T